# OsteoClassAI v2 · 06 Validación externa: Digital Knee X-ray Images (Gornale y Patravali)

Prueba de **robustez ante cambio de dominio**: 1650 radiografías de hospitales de la India (equipo PROTEC PRS 500E), graduadas por dos expertos. Mendeley Data, doi:10.17632/t9ndx37v5h.1, CC BY 4.0.

Los modelos se usan **congelados** (base, A y B), sin reentrenar. El protocolo quedó fijado antes de ver resultados (docstring de `osteoclass/external.py`): auditoría de duplicados, etiqueta del experto I sin conflictos, una imagen por rodilla, análisis principal con rodillas individuales rellenadas a cuadrado; sensibilidad con la imagen estirada; análisis secundario con las imágenes bilaterales.

**Antes de ejecutar:** suba `Digital Knee X-ray Images.zip` (el de Mendeley, 121 MB) a `MyDrive/OsteoClassAI/datos_externos/`.

*Entorno de ejecución → Cambiar tipo → GPU (T4) o CPU*, y luego *Ejecutar todas*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if os.path.isdir('/content/osteoclassai'):
    !cd /content/osteoclassai && git pull -q
else:
    !git clone -q https://github.com/walterandres23/osteoclassai.git /content/osteoclassai
%cd /content/osteoclassai/v2
!pip -q install -e . timm imagehash

In [ ]:
# Datos: el zip trae dos .rar (uno por experto)
ZIP = '/content/drive/MyDrive/OsteoClassAI/datos_externos/Digital Knee X-ray Images.zip'
!mkdir -p /content/gornale && cd /content/gornale && unzip -q -o "$ZIP"
!which unrar > /dev/null || apt-get -qq install -y unrar > /dev/null
!cd "/content/gornale/Digital Knee X-ray Images/Knee X-ray Images" && for f in *.rar; do unrar x -o+ -idq "$f"; done
root = '/content/gornale/Digital Knee X-ray Images/Knee X-ray Images'
!find "$root" -name '*.png' | wc -l   # debe dar 3300

## Evaluación de los tres modelos congelados

In [ ]:
RUNS = '/content/drive/MyDrive/OsteoClassAI/runs'
!python -m osteoclass.external --root "$root" --runs $RUNS/convnext_tiny $RUNS/convnext_tiny_sord $RUNS/convnext_tiny_sord_288 --out outputs/external/gornale
!mkdir -p /content/drive/MyDrive/OsteoClassAI/external && cp -r outputs/external/gornale /content/drive/MyDrive/OsteoClassAI/external/